# oF1 Score Computation with Hungarian Matching

This notebook computes the **Optimal F1 (oF1) score** for scientific image forgery detection using **Hungarian matching algorithm**. 

## What is oF1?
The oF1 metric evaluates instance-level forgery detection by:
1. Extracting predicted and ground truth instances using connected components
2. Matching instances using Hungarian algorithm based on IoU
3. Computing F1 score across multiple IoU thresholds
4. Taking the maximum F1 score (hence "optimal")

## System Requirements
- **Optimized for laptops with 16GB RAM**
- Batch processing to avoid memory issues
- Efficient numpy operations

## Authors
Abhishek Kumar, Ayomide Abayomi-Alli, Sinjini Ghosh

## 1. Import Required Libraries

In [ ]:
# Install required packages if not already installed
%pip install -q scipy pillow opencv-python tqdm seaborn matplotlib pandas numpy

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import linear_sum_assignment  # Hungarian algorithm
from scipy.ndimage import label as connected_components
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import gc  # Garbage collector for memory management

# For image processing
from PIL import Image
import cv2

print("✓ Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

✓ Libraries imported successfully!
NumPy version: 2.4.1
Pandas version: 3.0.0


## 2. Configuration and Helper Functions

In [ ]:
# Directory paths configured for local dataset
DATA_ROOT = Path("/home/abhishek/Deeplearning/recodai-luc-scientific-image-forgery-detection")

# IMPORTANT: You need to generate predictions first!
# Options:
# 1. Use predictions from your best model (e.g., DeepLabV3+ or U-Net)
# 2. Generate new predictions using the prediction generation cell below

# Predictions directory - UPDATE this after generating predictions
PRED_DIR = Path("/home/abhishek/Deeplearning/predictions")  # Will contain predicted masks (.npy files)

# Ground truth directory
GT_DIR = DATA_ROOT / "train_masks"

# Output directory for oF1 results
OUTPUT_DIR = Path("/home/abhishek/Deeplearning/oF1_results")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"✓ Data root: {DATA_ROOT}")
print(f"✓ Prediction directory: {PRED_DIR}")
print(f"✓ Ground truth directory: {GT_DIR}")
print(f"✓ Output directory: {OUTPUT_DIR}")
print(f"\nDataset structure:")
print(f"  - Train images: {len(list((DATA_ROOT / 'train_images' / 'authentic').glob('*'))) + len(list((DATA_ROOT / 'train_images' / 'forged').glob('*')))} files")
print(f"  - Train masks: {len(list(GT_DIR.glob('*.npy')))} files")
print(f"  - Predictions: {len(list(PRED_DIR.glob('*.npy')))} files")

if len(list(PRED_DIR.glob('*.npy'))) == 0:
    print("\n⚠️  WARNING: No predictions found!")
    print("   You need to generate predictions first using a trained model.")
    print("   See the prediction generation cell below.")

✓ Data root: /home/abhishek/Deeplearning/recodai-luc-scientific-image-forgery-detection
✓ Prediction directory: /home/abhishek/Deeplearning/recodai-luc-scientific-image-forgery-detection/train_masks
✓ Ground truth directory: /home/abhishek/Deeplearning/recodai-luc-scientific-image-forgery-detection/train_masks
✓ Output directory: oF1_results

Dataset structure:
  - Train images: 5128 files
  - Train masks: 2751 files


In [4]:
# Configuration
CONFIG = {
    'prediction_threshold': 0.5,  # Threshold for binarizing predictions
    'iou_thresholds': np.arange(0.5, 1.0, 0.05),  # IoU thresholds for oF1
    'batch_size': 10,  # Process images in batches to save memory
    'min_instance_size': 10,  # Minimum pixels for a valid instance
}

def compute_iou(mask1, mask2):
    """
    Compute Intersection over Union (IoU) between two binary masks.
    
    Args:
        mask1: Binary mask (numpy array)
        mask2: Binary mask (numpy array)
    
    Returns:
        float: IoU score
    """
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    
    if union == 0:
        return 0.0
    
    return intersection / union


def extract_instances(binary_mask, min_size=10):
    """
    Extract individual instances from a binary mask using connected components.
    
    Args:
        binary_mask: Binary mask (H, W)
        min_size: Minimum number of pixels for a valid instance
    
    Returns:
        list: List of binary masks for each instance
        int: Number of instances found
    """
    # Apply connected components labeling
    labeled_mask, num_instances = connected_components(binary_mask)
    
    instances = []
    for i in range(1, num_instances + 1):
        instance_mask = (labeled_mask == i)
        if instance_mask.sum() >= min_size:
            instances.append(instance_mask)
    
    return instances, len(instances)


def hungarian_matching(pred_instances, gt_instances, iou_threshold=0.5):
    """
    Match predicted instances to ground truth instances using Hungarian algorithm.
    
    Args:
        pred_instances: List of predicted instance masks
        gt_instances: List of ground truth instance masks
        iou_threshold: Minimum IoU for a valid match
    
    Returns:
        tuple: (true_positives, false_positives, false_negatives)
    """
    n_pred = len(pred_instances)
    n_gt = len(gt_instances)
    
    if n_pred == 0 and n_gt == 0:
        return 0, 0, 0
    
    if n_pred == 0:
        return 0, 0, n_gt
    
    if n_gt == 0:
        return 0, n_pred, 0
    
    # Compute IoU matrix (cost matrix)
    iou_matrix = np.zeros((n_pred, n_gt))
    for i, pred_mask in enumerate(pred_instances):
        for j, gt_mask in enumerate(gt_instances):
            iou_matrix[i, j] = compute_iou(pred_mask, gt_mask)
    
    # Hungarian algorithm expects cost (minimize), so we use negative IoU
    # Then we maximize IoU by minimizing -IoU
    cost_matrix = 1 - iou_matrix
    
    # Apply Hungarian matching
    pred_indices, gt_indices = linear_sum_assignment(cost_matrix)
    
    # Count matches above threshold
    true_positives = 0
    for i, j in zip(pred_indices, gt_indices):
        if iou_matrix[i, j] >= iou_threshold:
            true_positives += 1
    
    # Unmatched predictions are false positives
    false_positives = n_pred - true_positives
    
    # Unmatched ground truths are false negatives
    false_negatives = n_gt - true_positives
    
    return true_positives, false_positives, false_negatives


def compute_f1_score(tp, fp, fn):
    """
    Compute F1 score from TP, FP, FN counts.
    
    Args:
        tp: True positives
        fp: False positives
        fn: False negatives
    
    Returns:
        float: F1 score
    """
    if tp == 0:
        return 0.0
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    if precision + recall == 0:
        return 0.0
    
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1

print("✓ Helper functions defined successfully!")

✓ Helper functions defined successfully!


## 3. Load Predictions and Ground Truth

**Note:** Update the paths below to point to your actual prediction and ground truth directories.

In [ ]:
# Generate predictions from a trained model
# UNCOMMENT and RUN this cell to generate predictions from your best model

# Example: Load your best model and generate predictions
# import tensorflow as tf
# from tensorflow import keras
# 
# # Load your best model (update path)
# MODEL_PATH = "/home/abhishek/Deeplearning/data/unet_resnet50_bce_dice.keras"  # or deeplabv3plus model
# model = keras.models.load_model(MODEL_PATH, compile=False)
# 
# # Create prediction directory
# PRED_OUTPUT_DIR = Path("/home/abhishek/Deeplearning/predictions")
# PRED_OUTPUT_DIR.mkdir(exist_ok=True)
# 
# # Get list of validation/test images
# from PIL import Image
# import numpy as np
# 
# image_authentic = list((DATA_ROOT / "train_images" / "authentic").glob("*"))
# image_forged = list((DATA_ROOT / "train_images" / "forged").glob("*"))
# all_images = image_authentic + image_forged
# 
# TARGET_SIZE = (256, 256)  # Update based on your model's input size
# 
# print(f"Generating predictions for {len(all_images)} images...")
# 
# for img_path in tqdm(all_images, desc="Generating predictions"):
#     # Load and preprocess image
#     img = Image.open(img_path).convert('RGB')
#     img = img.resize(TARGET_SIZE)
#     img_array = np.array(img) / 255.0
#     img_array = np.expand_dims(img_array, axis=0)
#     
#     # Generate prediction
#     pred = model.predict(img_array, verbose=0)
#     pred_mask = (pred[0, :, :, 0] > 0.5).astype(np.uint8)  # Binary mask
#     
#     # Save prediction with same filename as ground truth mask
#     # Get corresponding mask filename from image path
#     mask_filename = img_path.stem + ".npy"
#     np.save(PRED_OUTPUT_DIR / mask_filename, pred_mask)
# 
# print(f"✓ Saved {len(all_images)} predictions to {PRED_OUTPUT_DIR}")

print("⚠️  Prediction generation is commented out.")
print("To generate predictions:")
print("1. Uncomment the code above")
print("2. Update MODEL_PATH to your best model")
print("3. Update TARGET_SIZE if needed")
print("4. Run this cell (may take 5-15 minutes)")
print("5. Then update PRED_DIR in the configuration cell above")

Prediction directory: predictions
Ground truth directory: ground_truth
⚠️  Prediction directory not found. Please update PRED_DIR path.
    Create dummy directory for demonstration purposes...
⚠️  Ground truth directory not found. Please update GT_DIR path.
    Create dummy directory for demonstration purposes...


## 4. Compute oF1 Score Across Dataset

This function processes all predictions and computes oF1 scores in batches to manage memory efficiently.

In [ ]:
def compute_oF1_dataset(pred_dir, gt_dir, iou_thresholds, batch_size=10):
    """
    Compute oF1 score across entire dataset with batch processing.
    
    Args:
        pred_dir: Directory with predicted masks (.npy files)
        gt_dir: Directory with ground truth masks (.npy files)
        iou_thresholds: Array of IoU thresholds to evaluate
        batch_size: Number of images to process before clearing memory
    
    Returns:
        dict: Results including oF1 score, F1 per threshold, etc.
    """
    # Get all prediction files (support .npy format)
    pred_files = sorted(list(pred_dir.glob("*.npy")))
    
    if len(pred_files) == 0:
        print("❌ ERROR: No prediction files (.npy) found!")
        return None
    
    print(f"Found {len(pred_files)} prediction files")
    
    # Initialize accumulators for each IoU threshold
    results_per_threshold = {
        iou_th: {'tp': 0, 'fp': 0, 'fn': 0} 
        for iou_th in iou_thresholds
    }
    
    # Process in batches
    num_batches = (len(pred_files) + batch_size - 1) // batch_size
    processed_count = 0
    
    for batch_idx in tqdm(range(num_batches), desc="Processing batches"):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, len(pred_files))
        batch_files = pred_files[start_idx:end_idx]
        
        for pred_file in batch_files:
            # Find corresponding ground truth file
            gt_file = gt_dir / pred_file.name
            
            if not gt_file.exists():
                print(f"⚠️  Missing ground truth for {pred_file.name}, skipping...")
                continue
            
            # Load masks (.npy format)
            pred_mask = np.load(pred_file)
            gt_mask = np.load(gt_file)
            
            # Binarize (in case they're not already binary)
            pred_binary = (pred_mask > 0.5).astype(np.uint8)
            gt_binary = (gt_mask > 0.5).astype(np.uint8)
            
            # Extract instances
            pred_instances, _ = extract_instances(pred_binary, CONFIG['min_instance_size'])
            gt_instances, _ = extract_instances(gt_binary, CONFIG['min_instance_size'])
            
            # Compute metrics for each IoU threshold
            for iou_th in iou_thresholds:
                tp, fp, fn = hungarian_matching(pred_instances, gt_instances, iou_th)
                results_per_threshold[iou_th]['tp'] += tp
                results_per_threshold[iou_th]['fp'] += fp
                results_per_threshold[iou_th]['fn'] += fn
            
            processed_count += 1
        
        # Clear memory after each batch
        gc.collect()
    
    # Compute F1 scores for each threshold
    f1_scores = []
    for iou_th in iou_thresholds:
        tp = results_per_threshold[iou_th]['tp']
        fp = results_per_threshold[iou_th]['fp']
        fn = results_per_threshold[iou_th]['fn']
        f1 = compute_f1_score(tp, fp, fn)
        f1_scores.append(f1)
    
    # Find optimal F1 (maximum across thresholds)
    oF1 = max(f1_scores)
    optimal_threshold_idx = np.argmax(f1_scores)
    optimal_threshold = iou_thresholds[optimal_threshold_idx]
    
    results = {
        'oF1': oF1,
        'optimal_iou_threshold': optimal_threshold,
        'f1_scores': f1_scores,
        'iou_thresholds': iou_thresholds,
        'results_per_threshold': results_per_threshold,
        'num_images': processed_count
    }
    
    return results

print("✓ oF1 computation function defined!")
print("  - Supports .npy mask files")
print("  - Batch processing for memory efficiency")
print("  - Ready to process your predictions")

✓ oF1 computation function defined!


## 5. Run oF1 Computation

**Execute this cell when you have actual predictions and ground truth masks ready.**

In [ ]:
# Check if predictions are ready
pred_files = list(PRED_DIR.glob("*.npy"))
gt_files = list(GT_DIR.glob("*.npy"))

if len(pred_files) == 0:
    print("❌ ERROR: No prediction files found!")
    print(f"   Please generate predictions first (see cell above)")
    print(f"   Expected directory: {PRED_DIR}")
elif len(gt_files) == 0:
    print("❌ ERROR: No ground truth files found!")
    print(f"   Expected directory: {GT_DIR}")
else:
    print(f"✓ Found {len(pred_files)} prediction files")
    print(f"✓ Found {len(gt_files)} ground truth files")
    print("\n🚀 Ready to run oF1 computation!")
    print("   Uncomment the code below and run this cell.")
    print(f"   Expected runtime: {len(pred_files) * 0.5 / 60:.1f}-{len(pred_files) * 2 / 60:.1f} minutes")
    
# UNCOMMENT TO RUN oF1 COMPUTATION
# results = compute_oF1_dataset(
#     pred_dir=PRED_DIR,
#     gt_dir=GT_DIR,
#     iou_thresholds=CONFIG['iou_thresholds'],
#     batch_size=CONFIG['batch_size']
# )
# 
# print("\n" + "="*60)
# print("oF1 COMPUTATION RESULTS")
# print("="*60)
# print(f"Optimal F1 Score (oF1): {results['oF1']:.4f}")
# print(f"Optimal IoU Threshold: {results['optimal_iou_threshold']:.2f}")
# print(f"Number of images evaluated: {results['num_images']}")
# print("="*60)
# 
# # Save detailed results
# results_df = pd.DataFrame({
#     'IoU_Threshold': results['iou_thresholds'],
#     'F1_Score': results['f1_scores']
# })
# results_df.to_csv(OUTPUT_DIR / 'oF1_scores.csv', index=False)
# print(f"\n✓ Saved detailed results to {OUTPUT_DIR / 'oF1_scores.csv'}")

## 6. Visualize F1 Scores Across IoU Thresholds

In [ ]:
# UNCOMMENT after running oF1 computation

# fig, ax = plt.subplots(1, 1, figsize=(10, 5))

# ax.plot(results['iou_thresholds'], results['f1_scores'], 
#         marker='o', linewidth=2.5, markersize=8, color='#2E86AB')
# ax.fill_between(results['iou_thresholds'], results['f1_scores'], 
#                  alpha=0.3, color='#2E86AB')

# # Mark optimal point
# optimal_idx = np.argmax(results['f1_scores'])
# ax.scatter([results['optimal_iou_threshold']], [results['oF1']], 
#           s=200, color='red', zorder=5, marker='*', 
#           label=f"oF1 = {results['oF1']:.4f} @ IoU={results['optimal_iou_threshold']:.2f}")

# ax.axvline(results['optimal_iou_threshold'], color='red', 
#           linestyle='--', alpha=0.5, linewidth=2)

# ax.set_xlabel('IoU Threshold', fontsize=12, fontweight='bold')
# ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
# ax.set_title('F1 Score vs IoU Threshold (Hungarian Matching)', 
#             fontsize=14, fontweight='bold')
# ax.legend(loc='upper right', fontsize=11)
# ax.grid(alpha=0.3)

# plt.tight_layout()
# plt.savefig(OUTPUT_DIR / 'oF1_curve.png', dpi=300, bbox_inches='tight')
# plt.savefig(OUTPUT_DIR / 'oF1_curve.pdf', dpi=300, bbox_inches='tight')
# print(f"✓ Saved visualization: {OUTPUT_DIR / 'oF1_curve.png'}")
# plt.show()

print("⚠️  Visualization is commented out")
print("   Uncomment and run after oF1 computation completes")

## 7. Example: Visualize Instance Matching

This section shows how to visualize the Hungarian matching results for a single image.

In [ ]:
def visualize_matching_example(pred_mask, gt_mask, iou_threshold=0.5):
    """
    Visualize Hungarian matching results for a single image.
    
    Args:
        pred_mask: Predicted binary mask
        gt_mask: Ground truth binary mask
        iou_threshold: IoU threshold for matching
    """
    # Extract instances
    pred_instances, n_pred = extract_instances(pred_mask, CONFIG['min_instance_size'])
    gt_instances, n_gt = extract_instances(gt_mask, CONFIG['min_instance_size'])
    
    print(f"Found {n_pred} predicted instances and {n_gt} ground truth instances")
    
    # Create colored masks
    pred_colored = np.zeros((*pred_mask.shape, 3), dtype=np.uint8)
    gt_colored = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    
    # Color each instance differently
    colors = plt.cm.rainbow(np.linspace(0, 1, max(n_pred, n_gt)))
    
    for i, instance in enumerate(pred_instances):
        color = (np.array(colors[i][:3]) * 255).astype(np.uint8)
        pred_colored[instance] = color
    
    for i, instance in enumerate(gt_instances):
        color = (np.array(colors[i][:3]) * 255).astype(np.uint8)
        gt_colored[instance] = color
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].imshow(pred_colored)
    axes[0].set_title(f'Predicted Instances ({n_pred})', fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(gt_colored)
    axes[1].set_title(f'Ground Truth Instances ({n_gt})', fontweight='bold')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Compute and display matching stats
    tp, fp, fn = hungarian_matching(pred_instances, gt_instances, iou_threshold)
    f1 = compute_f1_score(tp, fp, fn)
    
    print(f"\nMatching Results (IoU threshold = {iou_threshold}):")
    print(f"  True Positives: {tp}")
    print(f"  False Positives: {fp}")
    print(f"  False Negatives: {fn}")
    print(f"  F1 Score: {f1:.4f}")

# Example usage (uncomment when you have actual masks)
# pred_example = np.array(Image.open(PRED_DIR / "example.png").convert('L')) > 127
# gt_example = np.array(Image.open(GT_DIR / "example.png").convert('L')) > 127
# visualize_matching_example(pred_example, gt_example)

print("✓ Visualization function defined. Use when you have actual mask data.")

## 8. Summary and Next Steps

### Key Points:
1. **Hungarian Matching**: Optimal assignment algorithm that matches predicted instances to ground truth based on IoU
2. **oF1 Score**: Maximum F1 score across multiple IoU thresholds (typically 0.5-0.95)
3. **Memory Efficient**: Batch processing ensures it runs smoothly on 16GB RAM laptops
4. **Instance-Level Evaluation**: More stringent than pixel-level metrics like Dice/IoU

### To Use This Notebook:
1. Update `PRED_DIR` and `GT_DIR` to point to your prediction and ground truth directories
2. Ensure masks are binary (0 or 255) PNG/JPG files with matching filenames
3. Uncomment the computation cell and run
4. Results will include oF1 score and F1 curve visualization

### Performance Tips for Your Laptop:
- **Batch size**: Adjust `CONFIG['batch_size']` (lower if you encounter memory issues)
- **Min instance size**: Increase `CONFIG['min_instance_size']` to filter tiny artifacts
- **Image resolution**: Consider downsampling very large images if needed
- **Progress tracking**: Use tqdm progress bars to monitor long computations

## 9. Quick Test with Sample Data

Before running on full dataset, test with a few samples to verify everything works.

In [ ]:
# Test: Load a sample mask to verify paths are correct
sample_masks = list(GT_DIR.glob("*.npy"))[:5]

if sample_masks:
    print(f"✓ Found {len(sample_masks)} sample masks")
    print(f"\nFirst 5 mask files:")
    for mask_path in sample_masks:
        mask = np.load(mask_path)
        print(f"  {mask_path.name}: shape={mask.shape}, dtype={mask.dtype}, "
              f"unique_values={np.unique(mask)}, size={mask.nbytes/1024:.1f}KB")
    
    # Visualize first sample
    first_mask = np.load(sample_masks[0])
    plt.figure(figsize=(6, 6))
    plt.imshow(first_mask, cmap='gray')
    plt.title(f"Sample Mask: {sample_masks[0].name}")
    plt.colorbar()
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "sample_mask_visualization.png", dpi=150, bbox_inches='tight')
    print(f"\n✓ Saved sample visualization to {OUTPUT_DIR / 'sample_mask_visualization.png'}")
    plt.show()
else:
    print("✗ No mask files found! Please update PRED_DIR and GT_DIR paths.")

## 10. Run oF1 Computation (Uncomment to Execute)

**IMPORTANT**: Before running, update `PRED_DIR` with your model's prediction output directory.
The current configuration uses ground truth for both predictions and ground truth (for testing purposes only).